## 1. Importación de Librerías

En esta sección importamos las herramientas necesarias para el tratamiento de datos, visualización y construcción del modelo de Deep Learning.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pathlib
import os
import matplotlib.pyplot as plt
from tensorflow.keras.utils import image_dataset_from_directory
from sklearn.metrics import classification_report, confusion_matrix

# Se importan las librerías esenciales:
# - tensorflow y keras para la construcción y entrenamiento de la red neuronal.
# - numpy para operaciones numéricas y manejo de arreglos.
# - pathlib y os para la manipulación de rutas de archivos y directorios.
# - matplotlib para la visualización de imágenes y gráficas de rendimiento.
# - image_dataset_from_directory para cargar las imágenes desde carpetas estructuradas.
# - sklearn.metrics para evaluar el desempeño del modelo con métricas y matrices de confusión.

# 2. Carga del Dataset

In [ ]:
# Definimos la URL del dataset de flores proporcionado por TensorFlow
url = "http://download.tensorflow.org/example_images/flower_photos.tgz"

# Descargamos y descomprimimos el archivo tgz
data_dir = tf.keras.utils.get_file(
    origin=url,
    fname="flower_photos",
    untar=True
)

# 🔥 Corrección clave: Establecemos la ruta completa al directorio de imágenes extraídas
data_dir = pathlib.Path(data_dir) / "flower_photos"

# Imprimimos la ruta final donde se encuentran las carpetas de clases
print("Ruta final:", data_dir)

## 3. Preparación del DataSet


In [ ]:
# Obtenemos los nombres de las clases listando los nombres de los subdirectorios
class_names = sorted([item.name for item in data_dir.glob('*') if item.is_dir()])
print("Clases:", class_names)

# Definimos parámetros básicos para la carga de imágenes: tamaño de lote y dimensiones
batch_size = 32
img_height = 180
img_width = 180

# Creamos el dataset de entrenamiento (80% de los datos)
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# Creamos el dataset de validación (20% restante)
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)



# 4. Clases del dataset

In [ ]:
# Obtenemos los nombres de las clases directamente del objeto dataset
class_names = train_ds.class_names
print("Clases:", class_names)

# 5. Manejo de datos

In [ ]:
# Configuración de rendimiento: AUTOTUNE permite que TensorFlow ajuste dinámicamente el uso de recursos
AUTOTUNE = tf.data.AUTOTUNE

# Aplicamos optimizaciones a los datasets:
# - cache(): Mantiene las imágenes en memoria después de cargarlas en la primera época.
# - shuffle(1000): Desordena los datos con un buffer de 1000 elementos.
# - prefetch(): Prepara el siguiente lote de datos mientras el modelo entrena el actual.
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Definimos una capa de normalización para reescalar los píxeles de [0, 255] a [0, 1]
normalization_layer = tf.keras.layers.Rescaling(1./255)

# Mapeamos la normalización a los conjuntos de datos
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

In [ ]:
# Redefinimos los datasets (posiblemente para limpiar transformaciones previas antes de visualización o re-entrenamiento)
batch_size = 32
img_height = 180
img_width = 180

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# Definimos el número de clases basándonos en las etiquetas encontradas
class_names = train_ds.class_names
num_classes = len(class_names)

print("Clases:", class_names)
print("Número de clases:", num_classes)

# Configuración de Data Augmentation para mejorar la generalización del modelo:
# Se aplican rotaciones, zooms y volteos aleatorios a las imágenes durante el entrenamiento.
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

# Visualizamos una muestra de las imágenes cargadas
plt.figure(figsize=(10, 10))

for images, labels in train_ds.take(1):

    # 🔧 Aseguramos el rango correcto [0,1] dividiendo por 255 si es necesario o usando clip
    # Aquí se visualizan las imágenes antes del entrenamiento
    images = tf.clip_by_value(images, 0.0, 1.0)

    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)

        plt.imshow(images[i].numpy())
        plt.title(class_names[labels[i]])
        plt.axis("off")

plt.tight_layout()
plt.show()


## 6. Modelo CNN


In [ ]:
# Definición de la arquitectura del modelo CNN (Red Neuronal Convolucional)
model = tf.keras.Sequential([

    # Capa de aumento de datos integrada en el modelo
    data_augmentation,

    # Capa de reescalado opcional (comentada si ya se normalizó el dataset previamente)
    # layers.Rescaling(1./255),

    # Bloque Convolucional 1: Extrae características básicas mediante 32 filtros
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(), # Reduce la dimensión espacial

    # Bloque Convolucional 2: Extrae características más complejas con 64 filtros
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),

    # Bloque Convolucional 3: Características de alto nivel con 128 filtros
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),

    # Capa de Dropout: Apaga aleatoriamente el 30% de neuronas para evitar el sobreajuste (overfitting)
    layers.Dropout(0.3),

    # Fase de Clasificación
    layers.Flatten(), # Aplana las características 2D a un vector 1D
    layers.Dense(128, activation='relu'), # Capa densa con activación ReLU
    layers.Dense(num_classes)  # Capa de salida con el número de clases (sin softmax para manejar logits)
])

# Compilamos el modelo definiendo el optimizador, la función de pérdida y la métrica a monitorear
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Nota: El segundo model.compile es redundante pero mantiene la lógica previa del cuaderno
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# 7. Entrenamiento

In [ ]:
# Definimos el número de épocas (iteraciones completas sobre el dataset)
epochs = 30

# Iniciamos el entrenamiento del modelo alimentando los datos y validando cada época
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

In [ ]:
# Extraemos las métricas de precisión y pérdida obtenidas durante el entrenamiento
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

# Definimos el rango de épocas para el eje X de las gráficas
epochs_range = range(epochs)

# Configuramos la visualización de gráficas de rendimiento
plt.figure(figsize=(12, 5))

# Gráfica de Precisión (Accuracy)
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Entrenamiento')
plt.plot(epochs_range, val_acc, label='Validación')
plt.legend(loc='lower right')
plt.title('Accuracy')

# Gráfica de Pérdida (Loss)
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Entrenamiento')
plt.plot(epochs_range, val_loss, label='Validación')
plt.legend(loc='upper right')
plt.title('Loss')

plt.show()

In [ ]:
# Evaluamos formalmente el modelo con el conjunto de datos de validación
loss, accuracy = model.evaluate(val_ds)

# Imprimimos los resultados finales de evaluación
print("Loss en validación:", loss)
print("Accuracy en validación:", accuracy)

In [ ]:
# Tomamos un lote de imágenes de validación y realizamos predicciones individuales
for images, labels in val_ds.take(1):
    predictions = model.predict(images)
    predicted_classes = np.argmax(predictions, axis=1) # Obtenemos el índice de la clase con mayor probabilidad

    # Mostramos los primeros 5 resultados
    for i in range(5):
        print("Real:", class_names[labels[i]])
        print("Predicho:", class_names[predicted_classes[i]])
        print("----")

In [ ]:
# Aplicamos la función softmax a las predicciones para obtener probabilidades interpretables
probabilities = tf.nn.softmax(predictions)

# Visualizamos una cuadrícula de 3x3 con imágenes y sus respectivas predicciones y niveles de confianza
plt.figure(figsize=(10, 10))

for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy())

    pred = np.argmax(predictions[i]) # Clase predicha
    confidence = np.max(tf.nn.softmax(predictions[i])) # Confianza de la predicción

    plt.title(f"{class_names[pred]} ({confidence:.2f})")
    plt.axis("off")

plt.show()

In [ ]:
# Importamos Seaborn para mejorar la visualización de estadísticas
import seaborn as sns

# Inicializamos listas para almacenar las etiquetas reales y predichas de todo el set de validación
y_true = []
y_pred = []

# Iteramos sobre todo el conjunto de validación para construir la matriz de confusión
for images, labels in val_ds:
    preds = model.predict(images)
    preds = np.argmax(preds, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(preds)

# Calculamos la matriz de confusión
cm = confusion_matrix(y_true, y_pred)

# Visualizamos la matriz de confusión mediante un mapa de calor (Heatmap)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()

In [ ]:
model.save("modelo_flores.h5")